<a href="https://colab.research.google.com/github/Fajar-Sarfraz/week_4/blob/main/home_price_Week_4.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [6]:
from google.colab import files
uploaded = files.upload()

Saving modified_data.csv to modified_data.csv


week 4

In [7]:
import pandas as pd
df = pd.read_csv('modified_data.csv')   # ya jo bhi filename hai upload ki
print(df.shape)
print(df.columns.tolist())

(4600, 18)
['date', 'price', 'bedrooms', 'bathrooms', 'sqft_living', 'sqft_lot', 'floors', 'waterfront', 'view', 'condition', 'sqft_above', 'sqft_basement', 'yr_built', 'yr_renovated', 'street', 'city', 'statezip', 'price_per_sqft']


clean


In [8]:
# Drop leakage + unusable columns
df = df.drop(columns=['price_per_sqft', 'street', 'statezip'])

# Drop invalid price rows
df = df[df['price'] > 0]

# Extract sale year
df['date'] = pd.to_datetime(df['date'])
df['sale_year'] = df['date'].dt.year
df = df.drop(columns=['date'])

# Remove outliers (IQR method, same as Week 3)
Q1 = df['price'].quantile(0.25)
Q3 = df['price'].quantile(0.75)
IQR = Q3 - Q1
upper_bound = Q3 + 3 * IQR
df = df[df['price'] <= upper_bound]

print(df.shape)
print(df.columns.tolist())

(4462, 15)
['price', 'bedrooms', 'bathrooms', 'sqft_living', 'sqft_lot', 'floors', 'waterfront', 'view', 'condition', 'sqft_above', 'sqft_basement', 'yr_built', 'yr_renovated', 'city', 'sale_year']


X/y split aur train/test split

In [9]:
from sklearn.model_selection import train_test_split

X = df.drop(columns=['price'])
y = df['price']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

print(X_train.shape, X_test.shape)

(3569, 14) (893, 14)


In [11]:
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler, OneHotEncoder

numeric_features = ['bedrooms', 'bathrooms', 'sqft_living', 'sqft_lot', 'floors',
                     'waterfront', 'view', 'condition', 'sqft_above', 'sqft_basement',
                     'yr_built', 'yr_renovated', 'sale_year']
categorical_features = ['city']

preprocessor = ColumnTransformer(transformers=[
    ('num', StandardScaler(), numeric_features),
    ('cat', OneHotEncoder(handle_unknown='ignore'), categorical_features)
])

In [12]:
df.to_csv('cleaned_house_data.csv', index=False)
print("Saved!")

Saved!


In [13]:
from google.colab import files
files.download('cleaned_house_data.csv')

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

cross-validation

In [14]:
from sklearn.model_selection import cross_val_score, KFold
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor
import numpy as np

# 5-fold cross-validation setup
kfold = KFold(n_splits=5, shuffle=True, random_state=42)

# Linear Regression pipeline
lr_pipeline = Pipeline(steps=[
    ('preprocessor', preprocessor),
    ('regressor', LinearRegression())
])

# Random Forest pipeline (default settings for now)
rf_pipeline = Pipeline(steps=[
    ('preprocessor', preprocessor),
    ('regressor', RandomForestRegressor(n_estimators=100, random_state=42))
])

# Cross-validate both — using R² as the scoring metric
lr_scores = cross_val_score(lr_pipeline, X_train, y_train, cv=kfold, scoring='r2')
rf_scores = cross_val_score(rf_pipeline, X_train, y_train, cv=kfold, scoring='r2')

print("Linear Regression CV R² scores:", lr_scores)
print(f"Linear Regression: {lr_scores.mean():.4f} ± {lr_scores.std():.4f}")
print()
print("Random Forest CV R² scores:", rf_scores)
print(f"Random Forest: {rf_scores.mean():.4f} ± {rf_scores.std():.4f}")

Linear Regression CV R² scores: [0.70355591 0.71220865 0.6422252  0.7092458  0.71530144]
Linear Regression: 0.6965 ± 0.0274

Random Forest CV R² scores: [0.69959916 0.69908184 0.62897378 0.70626292 0.6977705 ]
Random Forest: 0.6863 ± 0.0288


GridSearchCV

In [15]:
from sklearn.model_selection import GridSearchCV

param_grid = {
    'regressor__n_estimators': [50, 100, 200],
    'regressor__max_depth': [5, 10, 15, None],
    'regressor__min_samples_leaf': [1, 5, 10]
}

grid_search = GridSearchCV(
    rf_pipeline,
    param_grid,
    cv=kfold,
    scoring='r2',
    n_jobs=-1
)

grid_search.fit(X_train, y_train)

print("Best parameters:", grid_search.best_params_)
print("Best CV R² score:", grid_search.best_score_)

Best parameters: {'regressor__max_depth': None, 'regressor__min_samples_leaf': 1, 'regressor__n_estimators': 200}
Best CV R² score: 0.6878540323293272


tuned model

In [16]:
best_rf_model = grid_search.best_estimator_

from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
import numpy as np

y_pred = best_rf_model.predict(X_test)

mae = mean_absolute_error(y_test, y_pred)
rmse = np.sqrt(mean_squared_error(y_test, y_pred))
r2 = r2_score(y_test, y_pred)

print(f"Tuned Random Forest — Test Set Performance")
print(f"MAE:  {mae:,.2f}")
print(f"RMSE: {rmse:,.2f}")
print(f"R²:   {r2:.4f}")
print()
print("Train R²:", best_rf_model.score(X_train, y_train))
print("Test R²:", best_rf_model.score(X_test, y_test))

Tuned Random Forest — Test Set Performance
MAE:  97,138.33
RMSE: 146,989.69
R²:   0.6787

Train R²: 0.9577403243574013
Test R²: 0.6787447800047433


Gradient Boosting

In [17]:
from sklearn.ensemble import GradientBoostingRegressor

gb_pipeline = Pipeline(steps=[
    ('preprocessor', preprocessor),
    ('regressor', GradientBoostingRegressor(random_state=42))
])

gb_scores = cross_val_score(gb_pipeline, X_train, y_train, cv=kfold, scoring='r2')
print("Gradient Boosting CV R² scores:", gb_scores)
print(f"Gradient Boosting: {gb_scores.mean():.4f} ± {gb_scores.std():.4f}")

# Fit on full training set and check test performance
gb_pipeline.fit(X_train, y_train)
y_pred_gb = gb_pipeline.predict(X_test)

mae_gb = mean_absolute_error(y_test, y_pred_gb)
rmse_gb = np.sqrt(mean_squared_error(y_test, y_pred_gb))
r2_gb = r2_score(y_test, y_pred_gb)

print(f"\nGradient Boosting — Test Set Performance")
print(f"MAE:  {mae_gb:,.2f}")
print(f"RMSE: {rmse_gb:,.2f}")
print(f"R²:   {r2_gb:.4f}")
print("Train R²:", gb_pipeline.score(X_train, y_train))
print("Test R²:", gb_pipeline.score(X_test, y_test))

Gradient Boosting CV R² scores: [0.71380729 0.69649811 0.66417989 0.7020018  0.70544512]
Gradient Boosting: 0.6964 ± 0.0171

Gradient Boosting — Test Set Performance
MAE:  99,336.99
RMSE: 145,800.35
R²:   0.6839
Train R²: 0.7648375793712902
Test R²: 0.6839224952293572


Feature Importance

In [18]:
# Feature names nikalna pipeline se
feature_names = (numeric_features +
                  list(gb_pipeline.named_steps['preprocessor']
                       .named_transformers_['cat']
                       .get_feature_names_out(categorical_features)))

importances = gb_pipeline.named_steps['regressor'].feature_importances_

import pandas as pd
feature_importance_df = pd.DataFrame({
    'feature': feature_names,
    'importance': importances
}).sort_values('importance', ascending=False)

print(feature_importance_df.head(10))

               feature  importance
2          sqft_living    0.559519
48        city_Seattle    0.076879
16       city_Bellevue    0.047325
8           sqft_above    0.045145
10            yr_built    0.039605
6                 view    0.037585
36  city_Mercer Island    0.031931
1            bathrooms    0.017847
44        city_Redmond    0.016002
31           city_Kent    0.015142


Final model

In [19]:
import joblib
joblib.dump(gb_pipeline, "final_house_price_model_week4.joblib")
print("Final Gradient Boosting model saved.")

# Best hyperparameters bhi record karein
print("\nGradient Boosting default hyperparameters used:")
print(gb_pipeline.named_steps['regressor'].get_params())

Final Gradient Boosting model saved.

Gradient Boosting default hyperparameters used:
{'alpha': 0.9, 'ccp_alpha': 0.0, 'criterion': 'friedman_mse', 'init': None, 'learning_rate': 0.1, 'loss': 'squared_error', 'max_depth': 3, 'max_features': None, 'max_leaf_nodes': None, 'min_impurity_decrease': 0.0, 'min_samples_leaf': 1, 'min_samples_split': 2, 'min_weight_fraction_leaf': 0.0, 'n_estimators': 100, 'n_iter_no_change': None, 'random_state': 42, 'subsample': 1.0, 'tol': 0.0001, 'validation_fraction': 0.1, 'verbose': 0, 'warm_start': False}


## Model Comparison — Week 3 Baseline vs Week 4 Tuning & Ensembles

| Model | CV Mean R² ± Std | Test R² | Train R² | Train/Test Gap |
|---|---|---|---|---|
| Linear Regression (Week 3 baseline) | — | 0.676 | 0.711 | 0.035 |
| Random Forest constrained (Week 3 baseline) | — | 0.658 | 0.799 | 0.141 |
| Linear Regression (CV) | 0.6965 ± 0.0274 | — | — | — |
| Random Forest default (CV) | 0.6863 ± 0.0288 | — | — | — |
| Random Forest tuned (GridSearchCV) | 0.6878 | 0.6787 | 0.9577 | 0.279 |
| **Gradient Boosting (final model)** | **0.6964 ± 0.0171** | **0.6839** | **0.765** | **0.081** |

**Final model: Gradient Boosting** — best test R², best RMSE, most stable cross-validation performance (lowest std across folds), and the smallest overfitting gap among all ensemble/tuned models. It outperforms the manually-constrained Week 3 Random Forest on every metric while requiring no manual tuning.

## Conclusion

This project extended the Week 3 house price prediction pipeline by applying
5-fold cross-validation, systematic hyperparameter search, and ensemble methods
to improve both performance and reliability of model evaluation.

Cross-validating the Week 3 baseline models revealed a key insight: Linear
Regression (0.6965 ± 0.0274) and default Random Forest (0.6863 ± 0.0288)
performed almost identically once evaluated across five folds rather than a
single split — resolving the unstable, seemingly contradictory train/test
results observed in Week 3. This confirmed that cross-validation gives a far
more trustworthy performance estimate than any single split.

GridSearchCV was used to tune the Random Forest across n_estimators, max_depth,
and min_samples_leaf. The search selected an unconstrained tree depth
(max_depth=None), which achieved the best CV score (0.6878) but reintroduced
substantial overfitting on the held-out test set (train R² 0.958 vs test R²
0.679, a 0.28 gap) — showing that optimizing for CV score alone does not
guarantee good generalization; the train/test gap must be checked separately.

Gradient Boosting, trained with default hyperparameters, outperformed every
other model: the highest and most stable cross-validation score (0.6964 ±
0.0171, the lowest std of any model), the best test R² (0.684), and by far
the smallest overfitting gap (0.081) of any tree-based model tested. Feature
importance confirmed sqft_living as the dominant predictor, followed by
high-value locations (Seattle, Bellevue, Mercer Island) — consistent with
real-world real estate fundamentals.

Limitations remain similar to Week 3: moderate dataset size and no
neighborhood-level features. Next steps include tuning Gradient Boosting's
own hyperparameters (learning_rate, max_depth, n_estimators) via
RandomizedSearchCV, and testing XGBoost for further gains.